<a href="https://colab.research.google.com/github/PauloRadatz/py_dss_toolkit/blob/master/examples/py-dss-toolkit_tutorial/11-Visualize_Feeder_Topology_user_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Circuit Interactive View (User-Defined Numerical and Categorical Plots)

This notebook demonstrates how to visualize a distribution feeder circuit using the `py_dss_toolkit` package. We’ll compile an OpenDSS model, solve a power flow, and generate interactive circuit plots for User-Defined Numerical and Categorical Plots.

**Contact:** paulo.radatz@gmail.com

If you’d like a structured learning path:
- **OpenDSS courses:** https://www.pauloradatz.me/opendss-courses
- **Learn the basics of controlling OpenDSS via Python (py-dss-interface course):** https://www.pauloradatz.me/course-py-dss-interface

## Install packages

We’ll install `py-dss-toolkit`. During installation, `pip` will also install `py-dss-interface`, which is a required dependency.

- **`py-dss-toolkit`**: utilities to run OpenDSS studies and generate high-level visualizations (including interactive circuit plots).
- **`py-dss-interface`**: a Python package that controls **OpenDSS Powered by EPRI** directly from Python.

In [1]:
!pip install py-dss-toolkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.9/121.9 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 41.5 MB/s eta 0:00:00


## Download the example feeder

Next, we’ll clone the `opendss-python-examples` repository, which includes ready-to-run OpenDSS feeder models.
We’ll use one of these feeders as the input circuit for the visualization examples in this notebook.

In [2]:
!git clone https://github.com/PauloRadatz/opendss-python-examples

Cloning into 'opendss-python-examples'...
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 31 (delta 3), reused 31 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (31/31), 176.48 KiB | 9.80 MiB/s, done.
Resolving deltas: 100% (3/3), done.


## Define the DSS master file path

Now we set the path to the **OpenDSS master file** (`Master_ckt5.dss`).
This file is the entry point of the feeder model—it typically redirects to all the other DSS files (lines, loads, transformers, etc.) needed to compile the circuit.

In [3]:
dss_file_path = "/content/opendss-python-examples/feeder_models/EPRITestCircuits/ckt5/Master_ckt5.dss"

## Imports

We’ll use `py-dss-interface` together with `py-dss-toolkit`:

- **`py_dss_interface`** provides a direct connection to the OpenDSS engine.
- **`dss_tools`** (from `py_dss_toolkit`) builds on top of that connection to simplify common workflows—such as accessing models, retrieving results, and creating interactive visualizations.

In [4]:
import py_dss_interface
from py_dss_toolkit import dss_tools

## Initialize OpenDSS and connect it to `dss_tools`

Next, we create a DSS engine instance using `py_dss_interface` to initialize the simulation environment.

Then we connect that same DSS instance to `dss_tools`, so all toolkit helper functions operate on the exact engine we just created.

In [5]:
dss = py_dss_interface.DSS()
dss_tools.update_dss(dss)
dss.started

True

## Compile the model and solve a power flow

Now we compile the OpenDSS master file and run a power flow solution:

- **`compile`** loads the circuit and all referenced files (via `Redirect` commands). For more details, see **Module 2** of: https://www.pauloradatz.me/course-snapshot
- **`solve`** runs the power flow so voltages, currents, and power flows are available for visualization and analysis.

In [6]:
dss.text(f"compile [{dss_file_path}]")
dss.text(f"solve")

''

## Plot losses in the circuit (User-Defined)

We can create user-defined numerical and categorical circuit plots by specifying custom results and settings.


In [7]:
dss_tools.interactive_view.user_numerical_defined_settings.results = dss_tools.results.losses_elements[0]["P losses (kW)"]
dss_tools.interactive_view.user_numerical_defined_settings.unit = "kW"
dss_tools.interactive_view.user_numerical_defined_settings.colorbar_title = "Losses in kW"
dss_tools.interactive_view.circuit_plot(parameter="user numerical defined", title="Losses per Line")

In [8]:
line_df = dss_tools.model.lines_df
line_df['name'] = 'line.' + line_df['name']
length = line_df.set_index("name")["length"].astype("float64")

threshold = 0.1
result = (length > threshold).astype(int)
result_df = (length > threshold).astype(int).to_frame("above_threshold")

dss_tools.interactive_view.user_categorical_defined_settings.results = result_df["above_threshold"]
dss_tools.interactive_view.user_categorical_defined_settings.color_map = {
    1: ["above_threshold", "blue"],
    0: ["below_threshold", "red"],
}

dss_tools.interactive_view.user_categorical_defined_settings.legendgrouptitle_text = f"Threshold={threshold}"
dss_tools.interactive_view.circuit_plot(parameter="user categorical defined", title="Line Length greater than threshold")

## Wrap-up

You’ve now seen a complete workflow to visualize an OpenDSS feeder with `py-dss-toolkit`:

- Install the packages and load an example feeder model
- Compile and solve a power flow with OpenDSS
- Create interactive circuit plots using User-Defined Numerical and Categorical Plots
- Add **bus markers** to highlight locations of interest

If you have questions, suggestions, or ideas for additional features/examples, feel free to reach out:

**Contact:** paulo.radatz@gmail.com

More learning resources:
- **OpenDSS courses:** https://www.pauloradatz.me/opendss-courses
- **Python + OpenDSS fundamentals (py-dss-interface course):** https://www.pauloradatz.me/course-py-dss-interface